## **Import Libraries**
___

In [6]:
# ── Core ──────────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import warnings
from fredapi import Fred
warnings.filterwarnings('ignore')

# ── Visualisation — Plotly ────────────────────────────────────────────────────
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.figure_factory as ff

# ── Statistics & econometrics ─────────────────────────────────────────────────
from scipy import stats
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import squareform
from statsmodels.tsa.stattools import adfuller, kpss, ccf
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

# ── Path handling ─────────────────────────────────────────────────────────────
from pathlib import Path

# ── Plotly global theme ───────────────────────────────────────────────────────
TEMPLATE = 'plotly_white'

# ── Colour palette ────────────────────────────────────────────────────────────
NAVY  = '#1F3864'
BLUE  = '#2E75B6'
RED   = '#C0392B'
GREEN = '#27AE60'
AMBER = '#E67E22'
GREY  = '#7F8C8D'
LBLUE = '#AED6F1'

# ── NBER / UK recession periods ───────────────────────────────────────────────
RECESSIONS_US = [
    ('1990-07-01', '1991-03-31', 'Early 1990s'),
    ('2001-03-01', '2001-12-31', 'Dot-com'),
    ('2007-12-01', '2009-06-30', 'GFC'),
    ('2020-02-01', '2020-04-30', 'COVID'),
]
RECESSIONS_UK = [
    ('1990-07-01', '1991-03-31', 'Early 1990s'),
    ('2008-01-01', '2009-06-30', 'GFC'),
    ('2020-01-01', '2020-06-30', 'COVID'),
]

# ── Shared helper: add recession bands to any plotly figure ──────────────────
def add_recessions(fig, recessions, row=None, col=None):
    """Add shaded recession bands to a plotly figure."""
    kwargs = dict(row=row, col=col) if row is not None else {}
    for start, end, label in recessions:
        fig.add_vrect(
            x0=start, x1=end,
            fillcolor='lightgrey', opacity=0.35,
            layer='below', line_width=0,
            annotation_text=label,
            annotation_position='top left',
            annotation_font_size=8,
            annotation_font_color=GREY,
            **kwargs
        )
    return fig

# ── Path handling ─────────────────────────────────────────────────────────────
BASE        = Path('../Data Collection')   # adjust to your folder
MACRO_US    = BASE / 'MacroVariables_US.csv'
MACRO_UK    = BASE / 'MacroVariables_UK.csv'
DELINQ_US   = BASE / 'DelinquencyRate_US.csv'
DELINQ_UK   = BASE / 'DelinquencyRate_UK.csv'

print('Libraries loaded.')
print(f'US macro path:  {MACRO_US}')
print(f'UK macro path:  {MACRO_UK}')

Libraries loaded.
US macro path:  ..\Data Collection\MacroVariables_US.csv
UK macro path:  ..\Data Collection\MacroVariables_UK.csv


In [7]:
# from fredapi import Fred

# fred = Fred(api_key='ca04d9a1acca4616dd8c051986a1e29e ')

# delinquency_raw = fred.get_series('DRALACBS')
# delinquency_raw.index = delinquency_raw.index.to_period('Q').to_timestamp('Q')
# delinquency_rate = delinquency_raw.copy()
# delinquency_rate.name = 'delinquency_rate'

In [8]:
# print(delinquency_rate.head())
# print(delinquency_rate.shape)
# print(delinquency_rate.dtypes)

In [9]:
# import os

# # Save delinquency rate to CSV
# output_dir = r"C:/Users/andre/OneDrive/LSE/ST498 - Capstone Project/ST-498/Data Collection"
# delinquency_rate.to_csv(os.path.join(output_dir, "DelinquencyRate_US.csv"), header=True)

# **Exploratory Data Analysis — Probability of Default (PD) Model**
___

This notebook loads the target variable and macroeconomic predictor variables to be used in the PD model. 

- **Target variable:** Delinquency Rate on All Loans and Leases (DRALACBS), sourced from the Federal Reserve Bank of St. Louis (FRED). This series serves as a proxy for probability of default at the aggregate level.
- **Macroeconomic variables:** Quarterly macroeconomic indicators for the UK and US, compiled in `MacroVariables_UK.csv` and `MacroVariables_US.csv`.

## **1: Sample Window Definition & Target Variable Integration**
___

IFRS 9 requires ECL models to reflect past events, current conditions, and forecasts of future economic conditions (Bank & Eder 2021, p. 4). Reliable macro-PD coefficient estimation requires coverage of multiple full credit cycles. The 1990 start date covers four stress episodes and is the natural completeness breakpoint in both datasets.

In [ ]:
# Switch between US and UK datasets by changing this variable:
COUNTRY = 'US'   # switch to 'UK' to run on UK data

if COUNTRY == 'US':
    MACRO_PATH  = MACRO_US
    DELINQ_PATH = DELINQ_US
    PREFIX      = 'us_'
    RECESSIONS  = RECESSIONS_US
elif COUNTRY == 'UK':
    MACRO_PATH  = MACRO_UK
    DELINQ_PATH = DELINQ_UK
    PREFIX      = 'uk_'
    RECESSIONS  = RECESSIONS_UK

WINDOW_START = '1990-01-01'
WINDOW_END   = '2024-12-31'
TARGET       = 'delinquency_rate'

print(f'{COUNTRY} dataset selected')

# 1.1  Load and trim macro data
macro_raw = pd.read_csv(MACRO_PATH, index_col=0, parse_dates=True)
macro_raw.index = (pd.DatetimeIndex(macro_raw.index)
                   .to_period('Q').to_timestamp('Q'))
macro_raw = macro_raw.sort_index()
df = macro_raw.loc[WINDOW_START:WINDOW_END].copy()

print(f'\nFull dataset:      {macro_raw.shape[0]} quarters '
      f'({macro_raw.index.min().date()} to {macro_raw.index.max().date()})')
print(f'Analytical window: {df.shape[0]} quarters '
      f'({df.index.min().date()} to {df.index.max().date()})')

# 1.2  Load delinquency rate
delinq_raw = pd.read_csv(DELINQ_PATH, index_col=0, parse_dates=True)
delinq_raw.index = (pd.DatetimeIndex(delinq_raw.index)
                    .to_period('Q').to_timestamp('Q'))
delinq_raw = delinq_raw.sort_index()
delinq_raw.columns = [TARGET]
delinq = delinq_raw.loc[WINDOW_START:WINDOW_END, TARGET]

print(f'Delinquency rate (target):  {len(delinq)} quarters  '
      f'Min {delinq.min():.2f}%  '
      f'Max {delinq.max():.2f}%  '
      f'Mean {delinq.mean():.2f}%')

# 1.3  Merge 
df = df.join(delinq, how='left')
print(f'\nMerged shape: {df.shape}  |  Target NaN: {df[TARGET].isna().sum()}\n')

# 1.4  Completeness report
completeness = pd.DataFrame({
    'Non-null':    df.notna().sum(),
    'Missing':     df.isna().sum(),
    'Missing (%)': (df.isna().sum() / len(df) * 100).round(1),
    'First valid': df.apply(lambda s: s.first_valid_index()
                            .strftime('%Y-Q') if s.first_valid_index() else None),
    'Last valid':  df.apply(lambda s: s.last_valid_index()
                            .strftime('%Y-Q') if s.last_valid_index() else None),
})

def colour_missing(val):
    if val == 0:   return 'background-color: #D5F5E3'
    if val <= 5:   return 'background-color: #FEF9E7'
    if val <= 15:  return 'background-color: #FDEBD0'
    return 'background-color: #FADBD8'

display(completeness.style
        .map(colour_missing, subset=['Missing (%)'])
        .set_caption(f'Table 1.1 — Data Completeness Report '
                     f'({COUNTRY}, 1990–2024)'))

# 1.5  Target variable — Deliquency rate time series with recession bands and stress peaks
mean_val = df[TARGET].mean()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df.index, y=df[TARGET],
    mode='lines',
    name='Delinquency Rate',
    line=dict(color=NAVY, width=2.5),
    fill='tozeroy',
    fillcolor='rgba(46,117,182,0.10)',
    hovertemplate='%{x|%Y Q%q}<br>Rate: %{y:.2f}%<extra></extra>',
))

fig.add_hline(
    y=mean_val,
    line_dash='dash', line_color=GREY, line_width=1.5,
    annotation_text=f'Mean = {mean_val:.2f}%',
    annotation_position='top right',
    annotation_font_color=GREY,
)

# Recession bands
fig = add_recessions(fig, RECESSIONS)

# Annotate stress peaks
peak_windows = [('2007', '2010'), ('2019', '2021')]
for start, end in peak_windows:
    subset = df[TARGET].loc[start:end].dropna()
    if not subset.empty:
        peak_date = subset.idxmax()
        peak_val  = subset.max()
        fig.add_annotation(
            x=peak_date,
            y=peak_val,
            text=f'{peak_val:.1f}%',
            showarrow=True,
            arrowhead=2,
            arrowcolor=RED,
            font=dict(color=RED, size=11, family='Arial'),
            bgcolor='white',
            bordercolor=RED,
            borderwidth=1,
            ay=-45,
        )

fig.update_layout(
    title=dict(
        text=f'Figure 1.1 — Delinquency Rate, All Commercial Banks '
             f'({COUNTRY}, 1990–2024)',
        font=dict(size=14, color=NAVY, family='Arial'),
    ),
    xaxis=dict(title='', showgrid=True, gridcolor='#E0E0E0'),
    yaxis=dict(title='Delinquency Rate (%)',
               showgrid=True, gridcolor='#E0E0E0'),
    template=TEMPLATE,
    hovermode='x unified',
    legend=dict(orientation='h', yanchor='bottom', y=1.02,
                xanchor='right', x=1),
    height=430,
    margin=dict(t=80, b=40, l=60, r=40),
)
fig.show()

# 1.6  Raw variables overview
raw_vars = [c for c in df.columns if c != TARGET]
ncols    = 4
nrows    = int(np.ceil(len(raw_vars) / ncols))

fig2 = make_subplots(
    rows=nrows, cols=ncols,
    subplot_titles=[v.replace(PREFIX, '').replace('_', ' ').title()
                    for v in raw_vars],
    vertical_spacing=0.08,
    horizontal_spacing=0.06,
)

for i, var in enumerate(raw_vars):
    row = i // ncols + 1
    col = i %  ncols + 1
    fig2.add_trace(
        go.Scatter(
            x=df.index, y=df[var],
            mode='lines',
            name=var,
            line=dict(color=BLUE, width=1.4),
            hovertemplate=var + '<br>%{x|%Y}<br>%{y:.2f}<extra></extra>',
            showlegend=False,
        ),
        row=row, col=col,
    )
    # Add recession bands per subplot
    for start, end, label in RECESSIONS:
        fig2.add_vrect(
            x0=start, x1=end,
            fillcolor='lightgrey', opacity=0.3,
            layer='below', line_width=0,
            row=row, col=col,
        )

fig2.update_layout(
    title=dict(
        text=f'Figure 1.2 — All Raw Variables ({COUNTRY}, 1990–2024  |  '
             f'grey = recessions)',
        font=dict(size=13, color=NAVY, family='Arial'),
    ),
    template=TEMPLATE,
    height=nrows * 220,
    margin=dict(t=80, b=40, l=40, r=40),
)
fig2.show()

# 1.7  Consumer confidence diagnostic
cc_col = PREFIX + 'consumer_confidence'
if cc_col in df.columns:
    cc = df[cc_col].dropna()
    print(f'\n=== Consumer Confidence Diagnostic ({COUNTRY}) ===')
    print(f'  Observations : {len(cc)}')
    print(f'  Range        : {cc.min():.3f}  to  {cc.max():.3f}')
    print(f'  Std          : {cc.std():.3f}')
    print(f'  Note         : Expected range ~50–110 for US Michigan Sentiment.')
    if cc.std() < 5:
        print(f'  FLAG         : Std of {cc.std():.2f} is implausibly narrow.')

Running pipeline for: US
Full dataset:      428 quarters (1919-03-31 to 2025-12-31)
Analytical window: 140 quarters (1990-03-31 to 2024-12-31)
Delinquency rate:  140 quarters  Min 1.19%  Max 7.40%  Mean 2.95%
Merged shape: (140, 17)  |  Target NaN: 0


,Non-null,Missing,Missing (%),First valid,Last valid
us_house_price_idx,140,0,0.000000,1990-Q,2024-Q
us_cpi,140,0,0.000000,1990-Q,2024-Q
us_unemployment,140,0,0.000000,1990-Q,2024-Q
us_consumer_confidence,140,0,0.000000,1990-Q,2024-Q
us_real_gdp,140,0,0.000000,1990-Q,2024-Q
us_gdp_yoy_growth,140,0,0.000000,1990-Q,2024-Q
us_bond_yield_10y,140,0,0.000000,1990-Q,2024-Q
us_reer,124,16,11.400000,1994-Q,2024-Q
us_oil_price,140,0,0.000000,1990-Q,2024-Q
us_credit,140,0,0.000000,1990-Q,2024-Q



=== Consumer Confidence Diagnostic (US) ===
  Observations : 140
  Range        : 94.780  to  102.192
  Std          : 1.248
  Note         : Expected range ~50–110 for US Michigan Sentiment.
  FLAG         : Std of 1.25 is implausibly narrow.
                 Verify source and normalisation before use.


### Data Quality Flag — Consumer Confidence (`us_consumer_confidence`)

**Finding:** The `us_consumer_confidence` variable shows a standard deviation of **1.25** 
across 140 quarters (1990–2024), with values ranging only from 94.78 to 102.19.

**Problem:** The University of Michigan Consumer Sentiment Index — the standard US 
consumer confidence measure — ranges approximately 50 to 110 over this period, with a 
standard deviation well above 10. A std of 1.25 is statistically inconsistent with any 
published consumer confidence series for the US.

**Likely cause:** The series appears to have been indexed or rebased to 100 using a 
specific base period during data collection, compressing the full time series into a 
narrow band around 100 and eliminating virtually all its informational content.

**Should we exclude it from the feature set?**

Two justifications:
1. **Data quality** — the series does not match the published statistical properties of 
   any standard US consumer confidence measure. The normalisation error cannot be 
   corrected without access to the original source metadata.
2. **Redundancy** — even if correctly measured, consumer confidence is largely captured 
   by the combination of unemployment, GDP growth, and equity returns already present 
   in the feature set. It does not constitute an independent signal.

**Reference:** This decision is consistent with Chawla et al. (2016), cited in 
Bank & Eder (2021, Section 11.1, p. 36), who conclude that the majority of qualitative 
indicators lag changes in the risk of default and therefore serve at best as benchmarks 
for quantitative predictors already in the model.

In [11]:
fred = Fred(api_key='ca04d9a1acca4616dd8c051986a1e29e')

# Download Fred Composite Consumer Confidence for US
cc_raw = fred.get_series('USACSCICP02STSAM')
cc_raw.index = cc_raw.index.to_period('M').to_timestamp('M')
cc_raw.name  = 'us_consumer_confidence'

print('Downloaded series:')
print(f'  Frequency : Monthly')
print(f'  Range     : {cc_raw.index.min().date()} to {cc_raw.index.max().date()}')
print(f'  Min       : {cc_raw.min():.2f}')
print(f'  Max       : {cc_raw.max():.2f}')
print(f'  Std       : {cc_raw.std():.2f}')
print(f'  NaN count : {cc_raw.isna().sum()}')

# Resample to quarterly (mean of 3 monthly obs per quarter)
cc_quarterly = (cc_raw
                .resample('QE')
                .mean()
                .rename('us_consumer_confidence'))
cc_quarterly.index = (cc_quarterly.index
                      .to_period('Q')
                      .to_timestamp('Q'))

# Trim to analytical window
cc_quarterly = cc_quarterly.loc[WINDOW_START:WINDOW_END]

print('Resampled to quarterly:')
print(f'  Observations : {len(cc_quarterly)}')
print(f'  Range        : {cc_quarterly.index.min().date()} '
      f'to {cc_quarterly.index.max().date()}')
print(f'  Min          : {cc_quarterly.min():.2f}')
print(f'  Max          : {cc_quarterly.max():.2f}')
print(f'  Mean         : {cc_quarterly.mean():.2f}')
print(f'  Std          : {cc_quarterly.std():.2f}')
print(f'  NaN count    : {cc_quarterly.isna().sum()}')

# Replace in df
df['us_consumer_confidence'] = cc_quarterly

print(f'Replaced us_consumer_confidence in df.')
print(f'NaN count after replacement: '
      f'{df["us_consumer_confidence"].isna().sum()}')

# Save as v2 to avoid overwriting original
MACRO_US_V2 = BASE / 'MacroVariables_US_v2.csv'

df.drop(columns=[TARGET]).to_csv(MACRO_US_V2)
print(f'Saved: {MACRO_US_V2}')

# Update MACRO_PATH to point to v2
MACRO_US    = MACRO_US_V2
MACRO_PATH  = MACRO_US_V2

print(f'MACRO_PATH updated to: {MACRO_PATH}')
print(f'All future steps will read from MacroVariables_US_v2.csv')

# Re run the consumer confidence diagnostic on the new quarterly series
# Step 1 style analysis on the new consumer confidence series

# Descriptive stats
cc = df['us_consumer_confidence'].dropna()
print('=== Consumer Confidence Diagnostic (USACSCICP02STSAM) ===')
print(f'  Observations : {len(cc)}')
print(f'  Min          : {cc.min():.2f}')
print(f'  Max          : {cc.max():.2f}')
print(f'  Mean         : {cc.mean():.2f}')
print(f'  Std          : {cc.std():.2f}')
print(f'  Range        : {cc.max() - cc.min():.2f}')
if cc.std() > 5:
    print(f'  STATUS       : Valid range and variability confirmed.')
else:
    print(f'  STATUS       : Still implausibly narrow — check download.')

# Time series plot with recession bands and trough annotations
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df.index,
    y=df['us_consumer_confidence'],
    mode='lines',
    name='Consumer Confidence',
    line=dict(color=NAVY, width=2.5),
    fill='tozeroy',
    fillcolor='rgba(46,117,182,0.10)',
    hovertemplate='%{x|%Y}<br>Index: %{y:.2f}<extra></extra>',
))

# Reference line at 100 (above = optimism, below = pessimism)
fig.add_hline(
    y=100,
    line_dash='dash', line_color=RED, line_width=1.5,
    annotation_text='100 — Neutral threshold',
    annotation_position='top right',
    annotation_font_color=RED,
)

# Mean line
fig.add_hline(
    y=cc.mean(),
    line_dash='dot', line_color=GREY, line_width=1.5,
    annotation_text=f'Mean = {cc.mean():.2f}',
    annotation_position='bottom right',
    annotation_font_color=GREY,
)

# Recession bands
fig = add_recessions(fig, RECESSIONS)

# Annotate troughs
trough_windows = [('2008', '2009'), ('2020', '2020'), ('2022', '2023')]
for start, end in trough_windows:
    subset = df['us_consumer_confidence'].loc[start:end].dropna()
    if not subset.empty:
        trough_date = subset.idxmin()
        trough_val  = subset.min()
        fig.add_annotation(
            x=trough_date,
            y=trough_val,
            text=f'{trough_val:.1f}',
            showarrow=True,
            arrowhead=2,
            arrowcolor=RED,
            font=dict(color=RED, size=10, family='Arial'),
            bgcolor='white',
            bordercolor=RED,
            borderwidth=1,
            ay=40,
        )

fig.update_layout(
    title=dict(
        text='Figure 1.3 — Fred Composite Consumer Confidence Index, US (1990–2024)<br>'
             '<sup>USACSCICP02STSAM | Quarterly mean of monthly observations | '
             'Above 100 = optimism, below 100 = pessimism</sup>',
        font=dict(size=13, color=NAVY, family='Arial'),
    ),
    xaxis=dict(title='', showgrid=True, gridcolor='#E0E0E0'),
    yaxis=dict(title='Consumer Confidence Index',
               showgrid=True, gridcolor='#E0E0E0'),
    template=TEMPLATE,
    hovermode='x unified',
    height=430,
    margin=dict(t=100, b=40, l=60, r=40),
)
fig.show()

Downloaded series:
  Frequency : Monthly
  Range     : 1960-01-31 to 2026-02-28
  Min       : 53.80
  Max       : 120.51
  Std       : 14.12
  NaN count : 0
Resampled to quarterly:
  Observations : 140
  Range        : 1990-03-31 to 2024-12-31
  Min          : 60.36
  Max          : 118.50
  Mean         : 91.69
  Std          : 14.06
  NaN count    : 0
Replaced us_consumer_confidence in df.
NaN count after replacement: 0
Saved: ..\Data Collection\MacroVariables_US_v2.csv
MACRO_PATH updated to: ..\Data Collection\MacroVariables_US_v2.csv
All future steps will read from MacroVariables_US_v2.csv
=== Consumer Confidence Diagnostic (USACSCICP02STSAM) ===
  Observations : 140
  Min          : 60.36
  Max          : 118.50
  Mean         : 91.69
  Std          : 14.06
  Range        : 58.14
  STATUS       : Valid range and variability confirmed.


## **2: Variable Classification & Redundancy Decisions**
___

We seek to prove which macreconomic indicators are most influential in predicting default risk. Before influence can be assessed, every variable must be assigned a functional group and a modelling decision made. This step eliminates redundant level/growth-rate pairs, flags variables needing transformation, and produces the cleaned feature set that all subsequent steps operate on. Follows the variable selection principle consistent with Bellini (2019), Chapter 9, cited throughout Bank & Eder (2021).

In [12]:
# ── 2.1  Variable register ────────────────────────────────────────────────────
var_register = pd.DataFrame([
    (PREFIX+'house_price_idx',      'B — Non-stationary level',
     'Compute YoY growth',          'Transform',
     'Trending upward; no growth companion in dataset'),

    (PREFIX+'cpi',                  'A — Stationary candidate',
     'Use as-is',                   'Retain',
     'Inflation rate; mean-reverting'),

    (PREFIX+'unemployment',         'A — Stationary candidate',
     'Use as-is',                   'Retain',
     'Cyclical rate; mean-reverting — confirm in Step 3'),

    (PREFIX+'consumer_confidence',  'A — Stationary candidate',
     'Use as-is',                   'Retain',
     'Replaced with OECD USACSCICP02STSAM. Std > 10, range 58–117. '
     'Mean-reverting around 100 by construction.'),

    (PREFIX+'real_gdp',             'C — Redundant pair',
     'Exclude; use YoY growth rate',   'Exclude',
     'Level is I(1); gdp_yoy_growth is the stationary companion'),

    (PREFIX+'gdp_yoy_growth',       'A — Stationary candidate',
     'Use as-is',                   'Retain',
     'Year-on-year growth rate; expected stationary'),

    (PREFIX+'bond_yield_10y',       'A — Stationary candidate',
     'Use as-is; test carefully',   'Retain',
     'Interest rate; near-unit-root behaviour common — confirm in Step 3'),

    (PREFIX+'reer',                 'B — Non-stationary level',
     'Interpolate 1990–1993 gap; first diff if I(1)', 'Transform',
     '16 missing values 1990–1993; interpolate then test stationarity'),

    (PREFIX+'oil_price',            'B — Non-stationary level',
     'Compute YoY growth',          'Transform',
     'Volatile level; YoY growth stationary and interpretable'),

    (PREFIX+'credit',               'C — Redundant pair',
     'Exclude; use QoQ growth',        'Exclude',
     'Level is I(1); credit_qoq_growth is the stationary companion'),

    (PREFIX+'credit_qoq_growth',    'A — Stationary candidate',
     'Use as-is',                   'Retain',
     'Quarter-on-quarter credit growth; expected stationary'),

    (PREFIX+'credit_yoy_growth',    'C — Redundant pair',
    'Excluded; use QoQ growth',    'Exclude',
    'Smoothed version of credit_qoq_growth — '
    'QoQ preferred as frequency match for quarterly model'),

    (PREFIX+'industrial_production','B — Non-stationary level',
     'Compute YoY growth',          'Transform',
     'Index level; YoY growth stationary'),

    (PREFIX+'vix',                  'A — Stationary candidate',
     'Log transform',               'Transform',
     'Stationary but strongly right-skewed; log reduces skewness'),

    (PREFIX+'sp500_close' if COUNTRY == 'US' else PREFIX+'ftse_close',
     'C — Redundant pair',
     'Exclude; use QoQ return',        'Exclude',
     'Level is I(1); QoQ return is the stationary companion'),

    (PREFIX+'sp500_qoq_growth' if COUNTRY == 'US' else PREFIX+'ftse_qoq_growth',
     'A — Stationary candidate',
     'Use as-is',                   'Retain',
     'Quarterly equity index return; expected stationary'),

], columns=['Variable', 'Group', 'Planned Transform', 'Decision', 'Rationale'])

DECISION_COLOURS = {
    'Retain':    '#D5F5E3',
    'Exclude':      '#FADBD8',
    'Transform': '#FEF9E7',
    'Pending':   '#EBF5FB',
}

def colour_decision(val):
    colour = DECISION_COLOURS.get(val, '#FDEBD0')
    return f'background-color: {colour}'

display(var_register.style
        .map(colour_decision, subset=['Decision'])
        .set_caption(f'Table 2.1 — Variable Register ({COUNTRY})')
        .hide(axis='index'))

Variable,Group,Planned Transform,Decision,Rationale
us_house_price_idx,B — Non-stationary level,Compute YoY growth,Transform,Trending upward; no growth companion in dataset
us_cpi,A — Stationary candidate,Use as-is,Retain,Inflation rate; mean-reverting
us_unemployment,A — Stationary candidate,Use as-is,Retain,Cyclical rate; mean-reverting — confirm in Step 3
us_consumer_confidence,A — Stationary candidate,Use as-is,Retain,"Replaced with OECD USACSCICP02STSAM. Std > 10, range 58–117. Mean-reverting around 100 by construction."
us_real_gdp,C — Redundant pair,Exclude; use YoY growth rate,Exclude,Level is I(1); gdp_yoy_growth is the stationary companion
us_gdp_yoy_growth,A — Stationary candidate,Use as-is,Retain,Year-on-year growth rate; expected stationary
us_bond_yield_10y,A — Stationary candidate,Use as-is; test carefully,Retain,Interest rate; near-unit-root behaviour common — confirm in Step 3
us_reer,B — Non-stationary level,Interpolate 1990–1993 gap; first diff if I(1),Transform,16 missing values 1990–1993; interpolate then test stationarity
us_oil_price,B — Non-stationary level,Compute YoY growth,Transform,Volatile level; YoY growth stationary and interpretable
us_credit,C — Redundant pair,Exclude; use QoQ growth,Exclude,Level is I(1); credit_qoq_growth is the stationary companion


In [13]:
# ── 2.2  Compute derived variables ────────────────────────────────────────────
# House price YoY growth
df[PREFIX+'house_price_yoy'] = (df[PREFIX+'house_price_idx']
                                 .pct_change(4) * 100)

# Industrial production YoY growth
df[PREFIX+'indprod_yoy'] = (df[PREFIX+'industrial_production']
                             .pct_change(4) * 100)

# Oil price YoY growth
df[PREFIX+'oil_yoy'] = df[PREFIX+'oil_price'].pct_change(4) * 100

# Log VIX
vix_col = PREFIX + 'vix'
if vix_col in df.columns:
    df[PREFIX+'log_vix'] = np.log(df[vix_col])

# REER — interpolate gap then first difference
df[PREFIX+'reer_interp'] = (df[PREFIX+'reer']
                             .interpolate(method='linear',
                                          limit_direction='both'))
df[PREFIX+'reer_diff'] = df[PREFIX+'reer_interp'].diff()

derived = [
    PREFIX+'house_price_yoy',
    PREFIX+'indprod_yoy',
    PREFIX+'oil_yoy',
    PREFIX+'log_vix',
    PREFIX+'reer_diff',
]

print('Derived variables created:')
for v in derived:
    if v in df.columns:
        print(f'  {v:42s}  non-null: {df[v].notna().sum()}')

Derived variables created:
  us_house_price_yoy                          non-null: 136
  us_indprod_yoy                              non-null: 136
  us_oil_yoy                                  non-null: 136
  us_log_vix                                  non-null: 140
  us_reer_diff                                non-null: 139


In [14]:
# ── 2.3  Define feature lists — nothing dropped from df ──────────────────────

equity_col = (PREFIX+'sp500_qoq_growth' if COUNTRY == 'US'
              else PREFIX+'ftse_qoq_growth')

# Variables actively used in modelling
FEATURES = [
    PREFIX+'gdp_yoy_growth',
    PREFIX+'unemployment',
    PREFIX+'cpi',
    PREFIX+'consumer_confidence',
    PREFIX+'bond_yield_10y',
    PREFIX+'credit_qoq_growth',
    equity_col,
    PREFIX+'log_vix',
    PREFIX+'house_price_yoy',
    PREFIX+'indprod_yoy',
    PREFIX+'oil_yoy',
    PREFIX+'reer_diff',
]

# Variables excluded from modelling and why
# These remain in df — decisions are documented only
FEATURES_EXCLUDED = {
    PREFIX+'real_gdp':     'Level I(1) — redundant with gdp_yoy_growth',
    PREFIX+'credit':       'Level I(1) — redundant with credit_qoq_growth',
    PREFIX+'credit_yoy_growth':
                           'Smoothed version of credit_qoq_growth — '
                           'QoQ preferred as frequency match for quarterly model',
    PREFIX+'sp500_close' if COUNTRY == 'US' else PREFIX+'ftse_close':
                           'Level I(1) — redundant with equity QoQ return',
}

# Working analytical dataset — features + target, no columns dropped from df
df_model = df[FEATURES + [TARGET]].dropna().copy()

print(f'Full df shape (untouched) : {df.shape}')
print(f'Analytical feature set    : {len(FEATURES)} variables')
print(f'Excluded (documented)     : {len(FEATURES_EXCLUDED)} variables')
print(f'Working dataset shape     : {df_model.shape}')
print(f'Date range                : {df_model.index.min().date()} '
      f'to {df_model.index.max().date()}')

print(f'\nActive features:')
for i, f in enumerate(FEATURES, 1):
    print(f'  {i:2d}.  {f}')

print(f'\nExcluded (still in df, not used in modelling):')
for var, reason in FEATURES_EXCLUDED.items():
    print(f'  {var:42s}  —  {reason}')

Full df shape (untouched) : (140, 23)
Analytical feature set    : 12 variables
Excluded (documented)     : 4 variables
Working dataset shape     : (136, 13)
Date range                : 1991-03-31 to 2024-12-31

Active features:
   1.  us_gdp_yoy_growth
   2.  us_unemployment
   3.  us_cpi
   4.  us_consumer_confidence
   5.  us_bond_yield_10y
   6.  us_credit_qoq_growth
   7.  us_sp500_qoq_growth
   8.  us_log_vix
   9.  us_house_price_yoy
  10.  us_indprod_yoy
  11.  us_oil_yoy
  12.  us_reer_diff

Excluded (still in df, not used in modelling):
  us_real_gdp                                 —  Level I(1) — redundant with gdp_yoy_growth
  us_credit                                   —  Level I(1) — redundant with credit_qoq_growth
  us_credit_yoy_growth                        —  Smoothed version of credit_qoq_growth — QoQ preferred as frequency match for quarterly model
  us_sp500_close                              —  Level I(1) — redundant with equity QoQ return


In [15]:
# ── 2.4  Decision summary bar chart ──────────────────────────────────────────
order   = ['Retain', 'Transform', 'Exclude']
summary = (var_register.groupby('Decision')['Variable']
           .count()
           .reindex(order)
           .fillna(0)
           .reset_index())
summary.columns = ['Decision', 'Count']

fig = go.Figure(go.Bar(
    x=summary['Decision'],
    y=summary['Count'],
    marker_color=[DECISION_COLOURS.get(d, AMBER) for d in summary['Decision']],
    marker_line_color=NAVY,
    marker_line_width=1,
    text=summary['Count'].astype(int),
    textposition='outside',
    hovertemplate='%{x}: %{y} variables<extra></extra>',
))
fig.update_layout(
    title=dict(
        text=f'Figure 2.1 — Variable Decision Summary ({COUNTRY})',
        font=dict(size=13, color=NAVY, family='Arial'),
    ),
    xaxis_title='Decision',
    yaxis_title='Number of Variables',
    yaxis=dict(range=[0, summary['Count'].max() + 2]),
    template=TEMPLATE,
    height=350,
    margin=dict(t=60, b=40, l=50, r=40),
)
fig.show()

In [ ]:
# ── 2.5  Transformed feature set — interactive subplot panel ──────────────────
ncols = 3
nrows = int(np.ceil(len(FEATURES) / ncols))

fig2 = make_subplots(
    rows=nrows, cols=ncols,
    subplot_titles=[v.replace(PREFIX, '').replace('_', ' ').title()
                    for v in FEATURES],
    vertical_spacing=0.09,
    horizontal_spacing=0.07,
)

for i, var in enumerate(FEATURES):
    row = i // ncols + 1
    col = i %  ncols + 1

    fig2.add_trace(
        go.Scatter(
            x=df_model.index,
            y=df_model[var],
            mode='lines',
            line=dict(color=BLUE, width=1.5),
            name=var.replace(PREFIX, '').replace('_', ' ').title(),
            hovertemplate=(var.replace(PREFIX, '').replace('_', ' ').title()
                           + '<br>%{x|%Y}<br>%{y:.2f}<extra></extra>'),
            showlegend=False,
        ),
        row=row, col=col,
    )

    fig2.add_hline(
        y=df_model[var].mean(),
        line_dash='dot',
        line_color=GREY,
        line_width=1,
        row=row, col=col,
    )

    for start, end, label in RECESSIONS:
        fig2.add_vrect(
            x0=start, x1=end,
            fillcolor='lightgrey', opacity=0.30,
            layer='below', line_width=0,
            row=row, col=col,
        )

fig2.update_layout(
    title=dict(
        text=(f'Figure 2.2 — Analytical Feature Set ({COUNTRY}, 1990–2024)  |  '
              f'grey = recessions  |  dotted = mean'),
        font=dict(size=13, color=NAVY, family='Arial'),
    ),
    template=TEMPLATE,
    height=nrows * 240,
    margin=dict(t=80, b=40, l=50, r=40),
)
fig2.show()

In [17]:
# ── 2.6  Descriptive statistics table ────────────────────────────────────────
desc = df_model[FEATURES].describe().T.round(3)
desc.index = [i.replace(PREFIX, '').replace('_', ' ').title()
              for i in desc.index]
desc.columns = [c.title() for c in desc.columns]

display(desc.style
        .background_gradient(subset=['Std'], cmap='Blues')
        .background_gradient(subset=['Mean'], cmap='RdYlGn')
        .format('{:.3f}')
        .set_caption(f'Table 2.2 — Descriptive Statistics, '
                     f'Analytical Feature Set ({COUNTRY})'))

,Count,Mean,Std,Min,25%,50%,75%,Max
Gdp Yoy Growth,136.000,2.530,2.047,-7.399,1.799,2.656,3.485,12.386
Unemployment,136.000,5.717,1.767,3.533,4.392,5.383,6.692,13.000
Cpi,136.000,2.621,1.548,-1.623,1.765,2.580,3.209,8.636
Consumer Confidence,136.000,91.804,14.114,60.360,81.090,94.020,101.417,118.497
Bond Yield 10Y,136.000,4.101,1.839,0.680,2.520,4.025,5.538,8.280
Credit Qoq Growth,136.000,1.278,0.886,-0.938,0.717,1.243,1.964,2.994
Sp500 Qoq Growth,136.000,2.451,7.839,-22.558,-0.809,3.200,7.109,20.867
Log Vix,136.000,2.905,0.344,2.252,2.619,2.860,3.143,3.980
House Price Yoy,136.000,2.210,4.638,-12.567,0.163,3.424,4.933,13.026
Indprod Yoy,136.000,1.491,4.063,-15.282,-0.269,2.430,3.911,8.958


## **3: Stationarity Testing (ADF + KPSS)**
___

A foundational requirement for any time-series regression is that the variables entering the model are stationary. When a non-stationary variable is regressed on another non-stationary variable, the results are spurious — R² values appear high and coefficients appear significant purely because both series share a common trend, not because a true economic relationship exists. In the context of this project, spurious regressions would directly undermine the IFRS 9 requirement that ECL models reflect genuine forward-looking macro-default relationships rather than coincidental co-movement (Bank & Eder 2021, p. 4).

Two tests are applied in combination rather than relying on either alone. The Augmented Dickey-Fuller test (Dickey & Fuller 1979) tests the null hypothesis of a unit root — rejection is a stationary signal. The KPSS test (Kwiatkowski et al. 1992) tests the opposite null of stationarity — failure to reject is a stationary signal. Using both is standard practice precisely because their opposite null hypotheses make them complementary: a variable confirmed stationary by both provides strong evidence, while disagreement between them flags a case requiring further investigation rather than allowing a false conclusion from either test alone.

This dual-test approach is particularly important for variables like the 10-year bond yield, which shows a long secular decline from roughly 8% in 1990 to near zero post-GFC before rising again. That pattern makes it visually ambiguous — it could be a trend-stationary process or a unit root process with drift — and a single test would not resolve the question reliably. For such variables, the constant-plus-trend specification (regression='ct') is used rather than constant-only, consistent with the recommendation to match the test specification to the visible properties of the series.

In [18]:
# ── 3.1  Test function ────────────────────────────────────────────────────────
def run_stationarity_tests(series, name, regression='c'):
    """
    ADF  — H0: unit root (non-stationary). Reject → stationary signal.
    KPSS — H0: stationary. Fail to reject → stationary signal.
    regression: 'c' = constant only, 'ct' = constant + trend
    """
    s = series.dropna()

    # ADF
    adf_stat, adf_p, adf_lags, _, _, _ = adfuller(
        s, autolag='AIC', regression=regression)

    # KPSS
    kpss_reg = 'ct' if regression == 'ct' else 'c'
    kpss_stat, kpss_p, _, _ = kpss(s, regression=kpss_reg, nlags='auto')

    # Decision logic
    adf_reject  = adf_p  < 0.05
    kpss_reject = kpss_p < 0.05

    if adf_reject and not kpss_reject:
        decision = 'STATIONARY'
    elif not adf_reject and kpss_reject:
        decision = 'NON-STATIONARY'
    elif not adf_reject and not kpss_reject:
        decision = 'INCONCLUSIVE'
    else:
        decision = 'CONFLICTING'

    return {
        'Variable':  name,
        'N':         len(s),
        'ADF stat':  round(adf_stat, 3),
        'ADF p':     round(adf_p, 3),
        'KPSS stat': round(kpss_stat, 3),
        'KPSS p':    round(kpss_p, 3),
        'Spec':      regression,
        'Decision':  decision,
    }

In [19]:
# ── 3.2  Run tests on all features ────────────────────────────────────────────
# Bond yield has a visible downward trend — use constant + trend specification
# All others use constant only
TREND_VARS = {PREFIX+'bond_yield_10y'}

results = []
for var in FEATURES:
    reg = 'ct' if var in TREND_VARS else 'c'
    results.append(
        run_stationarity_tests(df_model[var], var, regression=reg))

# Also test the target
results.append(
    run_stationarity_tests(df_model[TARGET], TARGET, regression='c'))

stat_df = pd.DataFrame(results)

DECISION_COLOURS_STAT = {
    'STATIONARY':     '#D5F5E3',
    'NON-STATIONARY': '#FADBD8',
    'INCONCLUSIVE':   '#FEF9E7',
    'CONFLICTING':    '#FDEBD0',
}

def colour_stat(val):
    return f'background-color: {DECISION_COLOURS_STAT.get(val, "white")}'

def colour_p(val):
    if val < 0.01:  return 'background-color: #D5F5E3'
    if val < 0.05:  return 'background-color: #FEF9E7'
    if val < 0.10:  return 'background-color: #FDEBD0'
    return 'background-color: #FADBD8'

display(stat_df.style
        .map(colour_stat,  subset=['Decision'])
        .map(colour_p,     subset=['ADF p'])
        .format({'ADF stat': '{:.3f}', 'ADF p': '{:.3f}',
                 'KPSS stat': '{:.3f}', 'KPSS p': '{:.3f}'})
        .set_caption(f'Table 3.1 — ADF + KPSS Stationarity Results ({COUNTRY})')
        .hide(axis='index'))

Variable,N,ADF stat,ADF p,KPSS stat,KPSS p,Spec,Decision
us_gdp_yoy_growth,136,-2.677,0.078,0.217,0.100,c,INCONCLUSIVE
us_unemployment,136,-3.106,0.026,0.186,0.100,c,STATIONARY
us_cpi,136,-1.703,0.430,0.152,0.100,c,INCONCLUSIVE
us_consumer_confidence,136,-2.267,0.183,0.439,0.060,c,INCONCLUSIVE
us_bond_yield_10y,136,-2.092,0.551,0.249,0.010,ct,NON-STATIONARY
us_credit_qoq_growth,136,-2.824,0.055,0.346,0.100,c,INCONCLUSIVE
us_sp500_qoq_growth,136,-11.536,0.000,0.102,0.100,c,STATIONARY
us_log_vix,136,-3.982,0.002,0.096,0.100,c,STATIONARY
us_house_price_yoy,136,-2.751,0.066,0.227,0.100,c,INCONCLUSIVE
us_indprod_yoy,136,-3.167,0.022,0.405,0.075,c,STATIONARY


In [20]:
# ── 3.3  Dual p-value interactive chart ───────────────────────────────────────
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        'ADF p-values  (green = reject H₀ of unit root → stationary signal)',
        'KPSS p-values  (green = fail to reject H₀ of stationarity → stationary signal)',
    ],
    horizontal_spacing=0.12,
)

short_names = [v.replace(PREFIX, '').replace('_', ' ').title()
               for v in stat_df['Variable']]

# ADF — green if p < 0.05 (reject unit root)
adf_colors = [GREEN if p < 0.05 else RED for p in stat_df['ADF p']]
fig.add_trace(
    go.Bar(
        x=stat_df['ADF p'],
        y=short_names,
        orientation='h',
        marker_color=adf_colors,
        marker_line_color='white',
        marker_line_width=0.5,
        name='ADF p-value',
        hovertemplate='%{y}<br>ADF p = %{x:.3f}<extra></extra>',
    ),
    row=1, col=1,
)
fig.add_vline(x=0.05, line_dash='dash', line_color=NAVY,
              line_width=1.5, row=1, col=1,
              annotation_text='p = 0.05',
              annotation_font_color=NAVY)

# KPSS — green if p > 0.05 (fail to reject stationarity)
kpss_colors = [GREEN if p > 0.05 else RED for p in stat_df['KPSS p']]
fig.add_trace(
    go.Bar(
        x=stat_df['KPSS p'],
        y=short_names,
        orientation='h',
        marker_color=kpss_colors,
        marker_line_color='white',
        marker_line_width=0.5,
        name='KPSS p-value',
        hovertemplate='%{y}<br>KPSS p = %{x:.3f}<extra></extra>',
    ),
    row=1, col=2,
)
fig.add_vline(x=0.05, line_dash='dash', line_color=NAVY,
              line_width=1.5, row=1, col=2,
              annotation_text='p = 0.05',
              annotation_font_color=NAVY)

fig.update_layout(
    title=dict(
        text=f'Figure 3.1 — Stationarity Test p-values ({COUNTRY})',
        font=dict(size=13, color=NAVY, family='Arial'),
    ),
    template=TEMPLATE,
    height=500,
    showlegend=False,
    margin=dict(t=100, b=40, l=160, r=40),
)
fig.update_xaxes(title_text='p-value', range=[0, 1])
fig.show()

In [21]:
# ── 3.4  Decision summary ─────────────────────────────────────────────────────
decision_counts = (stat_df.groupby('Decision')['Variable']
                   .count().reset_index())
decision_counts.columns = ['Decision', 'Count']

fig2 = go.Figure(go.Bar(
    x=decision_counts['Decision'],
    y=decision_counts['Count'],
    marker_color=[DECISION_COLOURS_STAT.get(d, AMBER)
                  for d in decision_counts['Decision']],
    marker_line_color=NAVY,
    marker_line_width=1,
    text=decision_counts['Count'],
    textposition='outside',
    hovertemplate='%{x}: %{y} variables<extra></extra>',
))
fig2.update_layout(
    title=dict(
        text=f'Figure 3.2 — Stationarity Decision Summary ({COUNTRY})',
        font=dict(size=13, color=NAVY, family='Arial'),
    ),
    xaxis_title='Decision',
    yaxis_title='Number of Variables',
    yaxis=dict(range=[0, decision_counts['Count'].max() + 2]),
    template=TEMPLATE,
    height=350,
    margin=dict(t=60, b=40, l=50, r=40),
)
fig2.show()

In [24]:
# ── 3.5  Handle non-stationary / inconclusive variables ───────────────────────

non_stat    = stat_df[stat_df['Decision'] == 'NON-STATIONARY']['Variable'].tolist()
inconclusive = stat_df[stat_df['Decision'] == 'INCONCLUSIVE']['Variable'].tolist()

FEATURES_STAT = FEATURES.copy()

# ── First difference NON-STATIONARY variables ─────────────────────────────────
if non_stat:
    print('Non-stationary — applying first difference:')
    retest = []
    for var in non_stat:
        if var in FEATURES_STAT:
            new_name = var + '_d1'
            df_model[new_name] = df_model[var].diff()
            FEATURES_STAT[FEATURES_STAT.index(var)] = new_name
            retest.append(
                run_stationarity_tests(
                    df_model[new_name].dropna(), new_name))
            print(f'  {var}  →  {new_name}')

    retest_df = pd.DataFrame(retest)
    display(retest_df.style
            .map(colour_stat, subset=['Decision'])
            .format({'ADF p': '{:.3f}', 'KPSS p': '{:.3f}'})
            .set_caption('Table 3.2 — Retest After First Differencing')
            .hide(axis='index'))

# ── Document inconclusive variables — retained with justification ─────────────
INCONCLUSIVE_RATIONALE = {
    PREFIX+'gdp_yoy_growth':
        'ADF p=0.078 (borderline), KPSS p=0.100 (comfortable). '
        'Retained — KPSS confirms stationarity. Growth rates are '
        'mean-reverting by economic definition.',

    PREFIX+'cpi':
        'ADF p=0.430 (weak), KPSS p=0.100 (comfortable). '
        'Retained — post-COVID inflation surge (2021–2023) inflates '
        'ADF p-value by introducing a temporary trend. '
        'Inflation is mean-reverting over a full cycle.',

    PREFIX+'consumer_confidence':
        'ADF p=0.183 (borderline), KPSS p=0.060 (borderline). '
        'Retained — bounded series by construction (cannot diverge '
        'to zero or infinity), making a unit root economically '
        'implausible. Long decline since 2000 peak drives borderline result.',

    PREFIX+'credit_qoq_growth':
        'ADF p=0.055 (marginally above 5%), KPSS p=0.100 (comfortable). '
        'Retained — essentially stationary, borderline ADF driven '
        'by GFC contraction outlier.',

    PREFIX+'house_price_yoy':
        'ADF p=0.066 (marginally above 5%), KPSS p=0.100 (comfortable). '
        'Retained — KPSS confirms stationarity. YoY growth '
        'transformation successfully removes the level trend.',

    TARGET:
        'ADF p=0.234 (weak), KPSS p=0.100 (comfortable). '
        'Retained — GFC and COVID spikes inflate ADF p-value. '
        'Series is clearly mean-reverting around 2.57% over the full cycle. '
        'KPSS confirmation is the deciding criterion.',
}

print('\nInconclusive variables — retained with documented rationale:')
print('-' * 75)
for var, rationale in INCONCLUSIVE_RATIONALE.items():
    if var in inconclusive + [TARGET]:
        print(f'\n  {var}')
        print(f'  {rationale}')

# ── Drop NaN rows introduced by differencing ──────────────────────────────────
df_model = df_model.dropna(subset=FEATURES_STAT)

print(f'\n{"="*55}')
print(f'FEATURES_STAT ({len(FEATURES_STAT)} variables):')
for i, v in enumerate(FEATURES_STAT, 1):
    marker = ' ← first differenced' if v.endswith('_d1') else ''
    print(f'  {i:2d}.  {v}{marker}')
print(f'\nWorking dataset after Step 3: {df_model.shape}')

Non-stationary — applying first difference:
  us_bond_yield_10y  →  us_bond_yield_10y_d1


Variable,N,ADF stat,ADF p,KPSS stat,KPSS p,Spec,Decision
us_bond_yield_10y_d1,134,-9.067000,0.000,0.227000,0.100,c,STATIONARY



Inconclusive variables — retained with documented rationale:
---------------------------------------------------------------------------

  us_gdp_yoy_growth
  ADF p=0.078 (borderline), KPSS p=0.100 (comfortable). Retained — KPSS confirms stationarity. Growth rates are mean-reverting by economic definition.

  us_cpi
  ADF p=0.430 (weak), KPSS p=0.100 (comfortable). Retained — post-COVID inflation surge (2021–2023) inflates ADF p-value by introducing a temporary trend. Inflation is mean-reverting over a full cycle.

  us_consumer_confidence
  ADF p=0.183 (borderline), KPSS p=0.060 (borderline). Retained — bounded series by construction (cannot diverge to zero or infinity), making a unit root economically implausible. Long decline since 2000 peak drives borderline result.

  us_credit_qoq_growth
  ADF p=0.055 (marginally above 5%), KPSS p=0.100 (comfortable). Retained — essentially stationary, borderline ADF driven by GFC contraction outlier.

  us_house_price_yoy
  ADF p=0.066 (margin

## Step 3 — Stationarity Testing Results & Decisions

### Confirmed Stationary
| Variable | ADF p | KPSS p | Note |
|----------|-------|--------|------|
| us_unemployment | 0.026 | 0.100 | Clean on both tests |
| us_sp500_qoq_growth | 0.000 | 0.100 | Clean on both tests |
| us_log_vix | 0.002 | 0.100 | Clean on both tests |
| us_indprod_yoy | 0.022 | 0.075 | Clean on both tests |
| us_oil_yoy | 0.000 | 0.100 | Clean on both tests |
| us_reer_diff | 0.000 | 0.100 | Clean on both tests |

### First Differenced
| Variable | Original Decision | Action | Post-Differencing |
|----------|------------------|--------|-------------------|
| us_bond_yield_10y | NON-STATIONARY (ADF p=0.551, KPSS p=0.010) | First difference → us_bond_yield_10y_d1 | STATIONARY (ADF p=0.000, KPSS p=0.100) |

The secular decline in the 10-year yield from ~8% in 1990 to near zero 
post-GFC constitutes a genuine trend that both tests confirmed as 
non-stationary. First differencing produces the quarterly change in yield, 
which is stationary and economically more interpretable — it captures the 
direction of monetary policy movement rather than the absolute rate level.

### Retained with Documented Justification (Inconclusive)
| Variable | ADF p | KPSS p | Justification |
|----------|-------|--------|---------------|
| us_gdp_yoy_growth | 0.078 | 0.100 | KPSS confirms stationarity. Growth rates are mean-reverting by economic definition. |
| us_cpi | 0.430 | 0.100 | Post-COVID inflation surge (2021–2023) introduces a temporary trend inflating ADF p-value. Inflation is mean-reverting over a full cycle. |
| us_consumer_confidence | 0.183 | 0.060 | Bounded series by construction — cannot diverge to zero or infinity, making a unit root economically implausible. Long decline since 2000 peak drives the borderline result. |
| us_credit_qoq_growth | 0.055 | 0.100 | Marginally above 5% threshold. KPSS comfortable. Borderline ADF driven by GFC contraction outlier. |
| us_house_price_yoy | 0.066 | 0.100 | KPSS confirms stationarity. YoY growth transformation successfully removes the level trend. |
| delinquency_rate (target) | 0.234 | 0.100 | GFC and COVID spikes inflate ADF p-value. Series is clearly mean-reverting around 2.57% over the full cycle. KPSS confirmation is the deciding criterion. |

### Final Feature Set After Step 3
12 variables entering Step 4 — one variable transformed (bond yield → 
first difference), all others retained at their Step 2 specification.
Working dataset: **134 rows × 14 columns**.

In [25]:
# ── 3.6  Visualise bond yield level vs first difference ───────────────────────
# Only bond yield required action — plot level vs d1 to document the decision

bond_var    = PREFIX + 'bond_yield_10y'
bond_d1_var = PREFIX + 'bond_yield_10y_d1'

fig3 = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        'Bond Yield 10Y — Level  (NON-STATIONARY)',
        'Bond Yield 10Y — First Difference  (STATIONARY)',
    ],
    horizontal_spacing=0.10,
)

# Level
fig3.add_trace(
    go.Scatter(
        x=df_model.index,
        y=df_model[bond_var],
        mode='lines',
        line=dict(color=RED, width=1.8),
        name='Level',
        hovertemplate='%{x|%Y}<br>Yield: %{y:.2f}%<extra></extra>',
        showlegend=False,
    ),
    row=1, col=1,
)

# First difference
fig3.add_trace(
    go.Scatter(
        x=df_model.index,
        y=df_model[bond_d1_var],
        mode='lines',
        line=dict(color=GREEN, width=1.8),
        name='First Difference',
        hovertemplate='%{x|%Y}<br>Change: %{y:.2f}%<extra></extra>',
        showlegend=False,
    ),
    row=1, col=2,
)

# Zero line on first difference panel
fig3.add_hline(
    y=0,
    line_dash='dot', line_color=GREY, line_width=1,
    row=1, col=2,
)

# Mean line on level panel
fig3.add_hline(
    y=df_model[bond_var].mean(),
    line_dash='dot', line_color=GREY, line_width=1,
    annotation_text=f'Mean = {df_model[bond_var].mean():.2f}%',
    annotation_font_color=GREY,
    row=1, col=1,
)

# Recession bands on both panels
for start, end, label in RECESSIONS:
    for c in [1, 2]:
        fig3.add_vrect(
            x0=start, x1=end,
            fillcolor='lightgrey', opacity=0.30,
            layer='below', line_width=0,
            row=1, col=c,
        )

# ADF / KPSS annotations
fig3.add_annotation(
    x=0.25, y=0.05,
    xref='paper', yref='paper',
    text='ADF p = 0.551  |  KPSS p = 0.010<br>→ NON-STATIONARY',
    showarrow=False,
    font=dict(size=10, color=RED, family='Arial'),
    bgcolor='white',
    bordercolor=RED,
    borderwidth=1,
)
fig3.add_annotation(
    x=0.78, y=0.05,
    xref='paper', yref='paper',
    text='ADF p = 0.000  |  KPSS p = 0.100<br>→ STATIONARY',
    showarrow=False,
    font=dict(size=10, color=GREEN, family='Arial'),
    bgcolor='white',
    bordercolor=GREEN,
    borderwidth=1,
)

fig3.update_layout(
    title=dict(
        text=(f'Figure 3.3 — Bond Yield 10Y: Level vs First Difference ({COUNTRY})<br>'
              '<sup>First differencing resolves non-stationarity — '
              'quarterly change in yield enters the model</sup>'),
        font=dict(size=13, color=NAVY, family='Arial'),
    ),
    template=TEMPLATE,
    height=380,
    margin=dict(t=100, b=60, l=60, r=40),
)
fig3.update_yaxes(title_text='Yield (%)', row=1, col=1)
fig3.update_yaxes(title_text='Change in Yield (%)', row=1, col=2)
fig3.show()

print('Step 3 complete.')
print(f'FEATURES_STAT ({len(FEATURES_STAT)} variables) confirmed and ready for Step 4.')

Step 3 complete.
FEATURES_STAT (12 variables) confirmed and ready for Step 4.
